# Poincaré Embeddings in Python: A Hierarchy in Two Dimensions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/deep-learning/hyperbolic_poincare_embeddings.ipynb)

Companion notebook to [the post](https://sesen.ai/blog/poincare-embeddings-python-hierarchy-two-dimensions).

A tree's node count grows exponentially with depth. The room inside a Euclidean
ball grows polynomially with radius. Hyperbolic space grows exponentially, which
is the shape a tree needs, so a hierarchy that takes many Euclidean dimensions
fits in a two-dimensional hyperbolic disc you can draw.

This notebook embeds the 1,170 WordNet mammal synsets in two numbers each,
measures the reconstruction against Euclidean embeddings from 2 to 200
dimensions, and shows where the geometry costs you: the rim of the disc, where
steps collapse and float32 runs out of mantissa.

CPU only, no GPU, no API key. The 2-D fit takes about a minute; the full
dimension ladder takes longer and is optional.

**Contents**

1. The mismatch, in exact arithmetic
2. The Poincaré ball
3. The hierarchy
4. Riemannian SGD
5. Reconstruction metrics
6. The two-dimensional fit
7. Depth, radius and the rim clamp
8. The dimension ladder
9. Where the rim costs you
10. GPT-2's vocabulary as a tree
11. Exercises

In [ ]:
!pip -q install nltk torch matplotlib

In [ ]:
import math
from collections import deque
from dataclasses import dataclass, field

import matplotlib.pyplot as plt
import torch

## 1. The mismatch, in exact arithmetic

A ternary tree has `3**d` nodes at depth `d`. A Euclidean circle of radius `d` has
circumference `2*pi*d`, linear in `d`. A hyperbolic circle of radius `d` has
circumference `2*pi*sinh(d)`, exponential in `d`.

Only one of those two keeps up with the tree.

In [ ]:
print(f"{'d':>3} {'3^d nodes':>12} {'Euclidean 2*pi*d':>18} {'hyperbolic 2*pi*sinh d':>24}")
for d in range(1, 9):
    print(f"{d:>3} {3 ** d:>12,} {2 * math.pi * d:>18.1f} {2 * math.pi * math.sinh(d):>24,.1f}")

Read the last two columns as room available at the same radius. By depth 8 the
tree needs 6,561 slots; the Euclidean circle offers 50 units of circumference and
the hyperbolic one offers 9,366. That gap is the entire idea.

## 2. The Poincaré ball

The model is the open unit disc, with distance

$$d(u,v) = \operatorname{arcosh}\left(1 + 2\frac{\|u-v\|^2}{(1-\|u\|^2)(1-\|v\|^2)}\right)$$

The two denominator terms are what make it hyperbolic. As a point approaches the
rim its $(1-\|x\|^2)$ goes to zero and every distance to it blows up, so the rim
is infinitely far away and there is unlimited room out there.

`riemannian_rescale` is the other half: the metric is the Euclidean one scaled by
$(2/(1-\|x\|^2))^2$, so a gradient in the ball picks up the inverse factor.

In [ ]:
BALL_EPS = 1e-5      # keep every point strictly inside the unit disc
ACOSH_EPS = 1e-7     # keep the arcosh argument strictly above 1

In [ ]:
def poincare_distance(u: torch.Tensor, v: torch.Tensor,
                      eps: float = BALL_EPS) -> torch.Tensor:
    """Geodesic distance in the Poincaré ball, broadcast over leading dims.

    d(u,v) = arcosh(1 + 2 |u-v|^2 / ((1-|u|^2)(1-|v|^2)))

    The denominator is what makes the geometry hyperbolic: as either point
    approaches the rim, its (1-|x|^2) term goes to zero and the distance blows up.
    """
    sq_u = u.pow(2).sum(-1).clamp(max=1 - eps)
    sq_v = v.pow(2).sum(-1).clamp(max=1 - eps)
    sq_diff = (u - v).pow(2).sum(-1)
    arg = 1 + 2 * sq_diff / ((1 - sq_u) * (1 - sq_v))
    return torch.acosh(arg.clamp(min=1 + ACOSH_EPS))


def euclidean_distance(u: torch.Tensor, v: torch.Tensor,
                       eps: float = BALL_EPS) -> torch.Tensor:  # noqa: ARG001
    return (u - v).pow(2).sum(-1).clamp(min=1e-12).sqrt()


def project_to_ball(x: torch.Tensor, eps: float = BALL_EPS) -> torch.Tensor:
    """Pull any point that has drifted outside the disc back to just inside it."""
    norm = x.norm(dim=-1, keepdim=True).clamp(min=1e-12)
    factor = torch.where(norm > 1 - eps, (1 - eps) / norm, torch.ones_like(norm))
    return x * factor


def riemannian_rescale(x: torch.Tensor) -> torch.Tensor:
    """The conformal factor that turns a Euclidean gradient into a Riemannian one.

    The Poincaré metric is the Euclidean one scaled by (2/(1-|x|^2))^2, so the
    gradient picks up the inverse: ((1-|x|^2)^2)/4. Points near the rim therefore
    take tiny steps, which is how the model stops itself falling off the edge.
    """
    return ((1 - x.pow(2).sum(-1, keepdim=True)) ** 2) / 4

In [ ]:
origin = torch.zeros(1, 2)
for r in (0.1, 0.5, 0.9, 0.99, 0.999):
    point = torch.tensor([[r, 0.0]])
    print(f"radius {r:<7} distance from centre {float(poincare_distance(origin, point)):7.3f}"
          f"   step multiplier {float(riemannian_rescale(point)):.3e}")

Each factor of ten closer to the rim buys about 2.3 more units of distance and
costs two orders of magnitude of step size. Both facts matter later.

## 3. The hierarchy

The benchmark from Nickel and Kiela: every synset under `mammal.n.01`, with the
**transitive closure** of the hypernym relation as the training set. A node is
related to every ancestor, not only its parent, which is what makes
reconstruction a test of whether the geometry can hold the hierarchy at all.

In [ ]:
import json
import pathlib
import nltk

nltk.download("wordnet", quiet=True)
CACHE = pathlib.Path("mammals.json")   # written on the first run, reused after

In [ ]:
@dataclass
class Hierarchy:
    """A rooted DAG, flattened into the arrays the training code wants."""

    name: str
    nodes: list[str]           # index -> label
    parents: list[list[int]]   # index -> direct parents (hypernyms inside the tree)
    closure: list[tuple[int, int]]  # (descendant, ancestor) over the whole closure
    depth: list[int]           # shortest hop count from the root

    @property
    def n(self) -> int:
        return len(self.nodes)

    def ancestors(self) -> list[set[int]]:
        out: list[set[int]] = [set() for _ in range(self.n)]
        for child, ancestor in self.closure:
            out[child].add(ancestor)
        return out

    def summary(self) -> str:
        return (f"{self.name}: {self.n} nodes, {len(self.closure)} closure pairs, "
                f"max depth {max(self.depth)}")


def _from_edges(name: str, nodes: list[str], parent_edges: set[tuple[str, str]]) -> Hierarchy:
    index = {label: i for i, label in enumerate(nodes)}
    parents: list[list[int]] = [[] for _ in nodes]
    children: list[list[int]] = [[] for _ in nodes]
    for child, parent in parent_edges:
        parents[index[child]].append(index[parent])
        children[index[parent]].append(index[child])

    roots = [i for i in range(len(nodes)) if not parents[i]]
    depth = [-1] * len(nodes)
    queue = deque(roots)
    for r in roots:
        depth[r] = 0
    while queue:
        node = queue.popleft()
        for child in children[node]:
            if depth[child] == -1:
                depth[child] = depth[node] + 1
                queue.append(child)

    closure: list[tuple[int, int]] = []
    for node in range(len(nodes)):
        seen: set[int] = set()
        stack = list(parents[node])
        while stack:
            ancestor = stack.pop()
            if ancestor in seen:
                continue
            seen.add(ancestor)
            stack.extend(parents[ancestor])
        closure.extend((node, ancestor) for ancestor in sorted(seen))

    return Hierarchy(name=name, nodes=nodes, parents=parents,
                     closure=sorted(closure), depth=depth)


def mammals(use_cache: bool = True) -> Hierarchy:
    """The transitive closure of the hypernym relation under `mammal.n.01`.

    Cached to JSON so the notebook runs without NLTK's corpus download.
    """
    if use_cache and CACHE.exists():
        blob = json.loads(CACHE.read_text())
        return _from_edges("mammals", blob["nodes"], {tuple(e) for e in blob["edges"]})

    from nltk.corpus import wordnet as wn  # noqa: PLC0415

    root = wn.synset("mammal.n.01")
    seen, edges = {root.name()}, set()
    stack = [root]
    while stack:
        synset = stack.pop()
        for hyponym in synset.hyponyms():
            edges.add((hyponym.name(), synset.name()))
            if hyponym.name() not in seen:
                seen.add(hyponym.name())
                stack.append(hyponym)

    nodes = sorted(seen)
    CACHE.parent.mkdir(parents=True, exist_ok=True)
    CACHE.write_text(json.dumps({"nodes": nodes, "edges": sorted(edges)}))
    return _from_edges("mammals", nodes, edges)


def balanced_tree(branching: int, depth: int) -> Hierarchy:
    """A perfect b-ary tree, where the node count at each level is known exactly."""
    nodes, edges = ["r"], set()
    frontier = ["r"]
    for _ in range(depth):
        nxt = []
        for parent in frontier:
            for b in range(branching):
                child = f"{parent}.{b}"
                nodes.append(child)
                edges.add((child, parent))
                nxt.append(child)
        frontier = nxt
    return _from_edges(f"tree-b{branching}-d{depth}", nodes, edges)

In [ ]:
tree = mammals()
print(tree.summary())
print("root:", tree.nodes[tree.depth.index(0)])
print("a leaf:", tree.nodes[tree.depth.index(max(tree.depth))])

## 4. Riemannian SGD

Three lines separate this from ordinary SGD:

1. rescale the gradient by $(1-\|\theta\|^2)^2/4$, so points near the rim take small steps
2. take the step
3. project anything that landed outside the disc back to just inside it

Drop the rescaling and points shoot past the rim on the first large gradient. Drop
the projection and the distance formula divides by a negative number.

The loss is the paper's: a softmax over the negative distances to one true
ancestor and 50 sampled non-ancestors, which pulls related nodes together and
pushes everything else away.

In [ ]:
@dataclass
class TrainConfig:
    dim: int = 2
    geometry: str = "hyperbolic"      # or "euclidean"
    epochs: int = 400
    burn_in: int = 20
    lr: float = 0.3
    negatives: int = 50
    batch_size: int = 512
    seed: int = 0
    snapshot_epochs: tuple[int, ...] = ()
    double: bool = False              # float64 instead of float32
    ball_eps: float = BALL_EPS        # how close to the rim a point may sit

    @property
    def dtype(self) -> torch.dtype:
        return torch.float64 if self.double else torch.float32


@dataclass
class TrainResult:
    embeddings: torch.Tensor
    losses: list[float] = field(default_factory=list)
    snapshots: dict[int, torch.Tensor] = field(default_factory=dict)
    seconds: float = 0.0


def _init_table(n: int, cfg: TrainConfig) -> torch.Tensor:
    generator = torch.Generator().manual_seed(cfg.seed)
    scale = 1e-3 if cfg.geometry == "hyperbolic" else 1e-1
    draw = torch.rand(n, cfg.dim, generator=generator, dtype=torch.float32)
    return ((draw * 2 - 1) * scale).to(cfg.dtype)


def train(hierarchy, cfg: TrainConfig) -> TrainResult:
    """Fit embeddings so that related pairs are closer than sampled unrelated ones."""
    import time  # noqa: PLC0415

    torch.manual_seed(cfg.seed)
    n = hierarchy.n
    distance = poincare_distance if cfg.geometry == "hyperbolic" else euclidean_distance

    edges = torch.tensor(hierarchy.closure, dtype=torch.long)
    related = torch.zeros(n, n, dtype=torch.bool)
    related[edges[:, 0], edges[:, 1]] = True
    related[torch.arange(n), torch.arange(n)] = True

    table = _init_table(n, cfg).requires_grad_(True)
    result = TrainResult(embeddings=table.detach().clone())
    start = time.time()

    for epoch in range(cfg.epochs):
        lr = cfg.lr / 10 if epoch < cfg.burn_in else cfg.lr
        order = torch.randperm(edges.shape[0])
        epoch_loss = 0.0

        for begin in range(0, edges.shape[0], cfg.batch_size):
            batch = edges[order[begin:begin + cfg.batch_size]]
            src, pos = batch[:, 0], batch[:, 1]
            negs = torch.randint(0, n, (batch.shape[0], cfg.negatives))
            targets = torch.cat([pos.unsqueeze(1), negs], dim=1)

            dist = distance(table[src].unsqueeze(1), table[targets], cfg.ball_eps)
            logits = -dist
            # a sampled "negative" that happens to be a true ancestor is dropped
            # rather than pushed away, which would fight the positive term
            collision = related[src.unsqueeze(1), targets]
            collision[:, 0] = False
            logits = logits.masked_fill(collision, float("-inf"))

            loss = torch.nn.functional.cross_entropy(
                logits, torch.zeros(batch.shape[0], dtype=torch.long))
            loss.backward()
            epoch_loss += loss.item() * batch.shape[0]

            with torch.no_grad():
                grad = table.grad
                if cfg.geometry == "hyperbolic":
                    grad = grad * riemannian_rescale(table)
                table -= lr * grad
                if cfg.geometry == "hyperbolic":
                    table.copy_(project_to_ball(table, cfg.ball_eps))
                table.grad = None

        result.losses.append(epoch_loss / edges.shape[0])
        if (epoch + 1) in cfg.snapshot_epochs:
            result.snapshots[epoch + 1] = table.detach().clone()

    result.embeddings = table.detach().clone()
    result.seconds = time.time() - start
    return result

## 5. Reconstruction metrics

For each node, rank each true ancestor against every node that is not an
ancestor. Mean rank of 1.0 means every ancestor is the nearest candidate. MAP
also punishes an embedding that gets one ancestor right and scatters the rest.

In [ ]:
@dataclass
class Reconstruction:
    mean_rank: float
    map_score: float


def evaluate(hierarchy, embeddings: torch.Tensor, geometry: str,
             ball_eps: float = BALL_EPS) -> Reconstruction:
    """Rank every true ancestor against every node that is not an ancestor.

    Mean rank of 1.0 means each true ancestor is the nearest candidate. MAP is the
    mean average precision of the true-ancestor set inside the full distance
    ranking, so it also punishes an embedding that gets one ancestor right and
    scatters the rest.
    """
    distance = poincare_distance if geometry == "hyperbolic" else euclidean_distance
    # in row blocks: the full broadcast is n x n x dim, which at dim 200 is several
    # gigabytes of intermediates and sends the machine to swap
    with torch.no_grad():
        n = embeddings.shape[0]
        matrix = torch.empty(n, n, dtype=embeddings.dtype)
        for begin in range(0, n, 64):
            block = embeddings[begin:begin + 64].unsqueeze(1)
            matrix[begin:begin + 64] = distance(block, embeddings.unsqueeze(0),
                                                ball_eps)
    matrix.fill_diagonal_(float("inf"))

    ancestors = hierarchy.ancestors()
    ranks: list[int] = []
    average_precisions: list[float] = []

    for node, truth in enumerate(ancestors):
        if not truth:
            continue
        row = matrix[node]
        truth_index = torch.tensor(sorted(truth), dtype=torch.long)
        # how many non-ancestors sit closer than each true ancestor
        closer = (row.unsqueeze(0) < row[truth_index].unsqueeze(1))
        closer[:, truth_index] = False
        closer[:, node] = False
        node_ranks = closer.sum(1) + 1
        ranks.extend(node_ranks.tolist())

        order = torch.argsort(row)
        hit = torch.zeros(len(row), dtype=torch.bool)
        hit[truth_index] = True
        hits = hit[order].float()
        precision = hits.cumsum(0) / torch.arange(1, len(row) + 1)
        average_precisions.append((precision * hits).sum().item() / len(truth))

    return Reconstruction(
        mean_rank=sum(ranks) / len(ranks),
        map_score=sum(average_precisions) / len(average_precisions),
    )


def fit_and_score(hierarchy, cfg: TrainConfig) -> tuple[TrainResult, Reconstruction]:
    result = train(hierarchy, cfg)
    return result, evaluate(hierarchy, result.embeddings, cfg.geometry, cfg.ball_eps)

## 6. The two-dimensional fit

About a minute on a laptop CPU. Watch the largest radius climb: the model pushes
the leaves outward on its own.

In [ ]:
cfg = TrainConfig(dim=2, geometry="hyperbolic", epochs=600, lr=2.0,
                  batch_size=64, seed=0)
fit, score = fit_and_score(tree, cfg)
print(f"mean rank {score.mean_rank:.2f}   MAP {score.map_score:.3f}   "
      f"{fit.seconds:.0f}s")
print(f"largest radius {float(fit.embeddings.norm(dim=1).max()):.6f}")

Drawing it needs one decision. In the raw coordinates a node at hyperbolic
distance 8.7 from the centre and one at 12.2 both land within a hundred-thousandth
of the rim, so the disc renders as a ring and the structure is invisible.
`rescale_radius` keeps every angle and replaces the drawn radius with hyperbolic
distance. No point moves relative to any other. Both views are below.

In [ ]:
import numpy as np

def rescale_radius(points):
    radius = np.linalg.norm(points, axis=1)
    distance = 2 * np.arctanh(np.clip(radius, 0, 1 - 1e-12))
    unit = np.where(radius[:, None] > 0, points / np.maximum(radius, 1e-12)[:, None], 0.0)
    return unit * (distance / distance.max())[:, None]


def draw_disc(ax, points, colours, title):
    ax.add_patch(plt.Circle((0, 0), 1.0, fill=False, color="#444", lw=1.4))
    for child, parents in enumerate(tree.parents):
        for parent in parents:
            ax.plot([points[parent, 0], points[child, 0]],
                    [points[parent, 1], points[child, 1]],
                    color="#9aa5ad", lw=0.4, alpha=0.2, zorder=1)
    art = ax.scatter(points[:, 0], points[:, 1], c=colours, cmap="viridis_r",
                     s=12, zorder=3)
    ax.set_xlim(-1.06, 1.06); ax.set_ylim(-1.06, 1.06)
    ax.set_aspect("equal"); ax.axis("off"); ax.set_title(title, fontsize=10.5)
    return art


raw = fit.embeddings.numpy()
depth = torch.tensor(tree.depth).numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 6.4))
draw_disc(axes[0], raw, depth, "as the coordinates sit")
art = draw_disc(axes[1], rescale_radius(raw), depth,
                "radius rescaled to hyperbolic distance")
fig.colorbar(art, ax=axes, fraction=0.022).set_label("depth below mammal.n.01")
fig.suptitle("1,170 mammal synsets, two numbers each")
plt.show()

## 7. Depth, radius and the rim

Nothing in the loss mentions depth. The training set is a bag of
(node, ancestor) pairs with no level attached, so any radial ordering the model
produces is a side effect of packing: a node with many descendants needs room
around it, and room is what the region near the rim has.

The ordering does appear, and then a long run destroys it. `BALL_EPS` caps the
distance from the centre at `2 artanh(1 - 1e-5)`, about 12.21, and the loss has
no reason to stop pushing, so almost everything ends up at that ceiling. Print
the medians in **hyperbolic** distance rather than raw radius: in raw radius
every level below the root reads as 0.99999 and the structure is invisible.

In [ ]:
origin = torch.zeros(1, 2)
from_origin = poincare_distance(origin, fit.embeddings, 1e-16)
ceiling = 2 * math.atanh(1 - 1e-5)

for level in sorted(set(tree.depth)):
    picked = torch.tensor([i for i, d in enumerate(tree.depth) if d == level])
    print(f"depth {level}: {len(picked):>4} nodes, median radius "
          f"{float(fit.embeddings.norm(dim=1)[picked].median()):.5f}, "
          f"median distance {float(from_origin[picked].median()):6.2f}")

depths = torch.tensor(tree.depth, dtype=torch.float32)
cd, cn = depths - depths.mean(), from_origin - from_origin.mean()
print(f"\nceiling set by the rim clamp: {ceiling:.2f}")
print(f"correlation of depth with distance: {float((cd * cn).sum() / (cd.norm() * cn.norm())):.3f}")

Score the snapshots and the trade-off becomes explicit: reconstruction improves
all the way to epoch 600 while the radial ordering peaks around epoch 200 and
then loses nearly half its strength. Angle is the durable signal; radius is a
trend with an expiry date.

In [ ]:
snap_cfg = TrainConfig(dim=2, geometry="hyperbolic", epochs=600, lr=2.0,
                       batch_size=64, seed=0,
                       snapshot_epochs=(25, 50, 100, 200, 350, 600))
snapped = train(tree, snap_cfg)

print(f"{'epoch':>6} {'mean rank':>10} {'MAP':>7} {'depth corr':>11} {'median dist':>12}")
for epoch in sorted(snapped.snapshots):
    points = snapped.snapshots[epoch]
    s = evaluate(tree, points, "hyperbolic")
    d = poincare_distance(origin, points, 1e-16)
    cv = d - d.mean()
    corr = float((cd * cv).sum() / (cd.norm() * cv.norm()))
    print(f"{epoch:>6} {s.mean_rank:>10.2f} {s.map_score:>7.3f} {corr:>11.3f} "
          f"{float(d.median()):>12.2f}")

## 8. The dimension ladder

The honest comparison: same loss, same negatives, same schedule, only the
distance function changes. This cell takes several minutes. Trim the list to
taste.

Two results come out of it. Five hyperbolic dimensions beat two hundred
Euclidean ones, 1.15 against 1.47 on mean rank. Two hyperbolic dimensions land
around twenty Euclidean ones, which is a band rather than a point: the hyperbolic
seeds vary by 1.02 in mean rank, and at a longer budget Euclidean 20 takes the
lead. The gap widens with the tree, so measure it on yours.

In [ ]:
ladder = [("hyperbolic", 2), ("hyperbolic", 5), ("euclidean", 10),
          ("euclidean", 20), ("euclidean", 50), ("euclidean", 200)]

for geometry, dim in ladder:
    cfg = TrainConfig(dim=dim, geometry=geometry, epochs=600, lr=2.0,
                      batch_size=64, seed=0)
    _, s = fit_and_score(tree, cfg)
    print(f"{geometry:<11} d={dim:<4} mean rank {s.mean_rank:8.2f}   MAP {s.map_score:.3f}")

## 9. Where the rim costs you

Two costs, both real in production code.

**The step collapses.** The rescaling that keeps points inside the disc also means
a point at radius 0.999 moves about 250,000 times less per unit of Euclidean
gradient than one at the centre. Leaves settle early and stop refining.

**float32 runs out of mantissa.** The distance formula needs `1 - r**2`. In
float32 the largest value below 1.0 is about `1 - 6e-8`, so at a gap of `1e-8`
the term is exactly zero and the distance is infinite. Every implementation
clamps the radius, and the clamp is what caps the disc.

In [ ]:
print(f"{'gap to rim':>12} {'1-r^2 (f32)':>14} {'d(r,-r) f32':>14} {'d(r,-r) f64':>14} {'exact':>10}")
for gap in (1e-4, 1e-6, 1e-7, 3e-8, 1e-8):
    r = 1 - gap
    a32 = torch.tensor([[r, 0.0], [-r, 0.0]], dtype=torch.float32)
    a64 = torch.tensor([[r, 0.0], [-r, 0.0]], dtype=torch.float64)
    print(f"{gap:>12.0e} {float(1 - torch.tensor(r, dtype=torch.float32) ** 2):>14.3e} "
          f"{float(poincare_distance(a32[0], a32[1], 1e-16)):>14.4f} "
          f"{float(poincare_distance(a64[0], a64[1], 1e-16)):>14.4f} "
          f"{4 * math.atanh(r):>10.4f}")

`BALL_EPS` is the clamp, and moving it is worth real accuracy. Compare the two
float32 rows below: the same model and the same data, with the clamp moved from
the paper's `1e-5` to the `1e-7` that float32 can still represent. The float64
rows show the gain comes from the extra room and not from the precision, since
float64 at the same clamp does no better.

In [ ]:
for double, eps in ((False, 1e-5), (False, 1e-7), (True, 1e-5), (True, 1e-9)):
    cfg = TrainConfig(dim=2, geometry="hyperbolic", epochs=600, lr=2.0,
                      batch_size=64, seed=0, double=double, ball_eps=eps)
    r, s = fit_and_score(tree, cfg)
    print(f"{'float64' if double else 'float32'}  eps={eps:<8g} "
          f"mean rank {s.mean_rank:6.2f}  MAP {s.map_score:.3f}  "
          f"max radius {float(r.embeddings.norm(dim=1).max()):.8f}")

## 10. GPT-2's vocabulary as a tree

A tokeniser vocabulary carries a hierarchy nobody designed: the character prefix
tree over its tokens. Every proper prefix is a node, and a token's parent is the
prefix one character shorter. The same code embeds it without changes.

In [ ]:
def gpt2_prefix_tree(vocabulary: int = 1000) -> Hierarchy:
    """The character prefix tree of GPT-2's most common word-initial tokens.

    BPE assigns low ids to the merges it learned first, which are the frequent
    ones, so taking the first `vocabulary` space-prefixed alphabetic tokens gives
    the common end of the vocabulary. Every proper prefix of a token becomes a
    node, and a token's parent is the prefix one character shorter.
    """
    from transformers import AutoTokenizer  # noqa: PLC0415

    tokenizer = AutoTokenizer.from_pretrained("gpt2")
    by_id = {i: t for t, i in tokenizer.get_vocab().items()}

    words: list[str] = []
    for token_id in range(len(by_id)):
        token = by_id[token_id]
        body = token[1:]
        if token.startswith("Ġ") and len(body) > 1 and body.isalpha() and body.islower():
            words.append(body)
        if len(words) == vocabulary:
            break

    nodes, edges = {""}, set()
    for word in words:
        for cut in range(1, len(word) + 1):
            nodes.add(word[:cut])
            edges.add((word[:cut], word[:cut - 1]))

    return _from_edges("gpt2-prefixes", sorted(nodes), edges)

In [ ]:
prefixes = gpt2_prefix_tree(1000)
print(prefixes.summary())

cfg = TrainConfig(dim=2, geometry="hyperbolic", epochs=600, lr=2.0,
                  batch_size=64, seed=0)
gpt2_fit, gpt2_score = fit_and_score(prefixes, cfg)
print(f"mean rank {gpt2_score.mean_rank:.2f}   MAP {gpt2_score.map_score:.3f}")

points = gpt2_fit.embeddings.numpy()
fig, ax = plt.subplots(figsize=(7, 7))
ax.add_patch(plt.Circle((0, 0), 1.0, fill=False, color="#444", lw=1.4))
for child, parents in enumerate(prefixes.parents):
    for parent in parents:
        ax.plot([points[parent, 0], points[child, 0]],
                [points[parent, 1], points[child, 1]],
                color="#9aa5ad", lw=0.4, alpha=0.2, zorder=1)
ax.scatter(points[:, 0], points[:, 1],
           c=torch.tensor(prefixes.depth).numpy(), cmap="viridis_r", s=10, zorder=3)
for label in ("c", "co", "con", "s", "st", "p", "th", "the", "a", "in", "re"):
    if label in prefixes.nodes:
        x, y = points[prefixes.nodes.index(label)]
        ax.annotate(label, (x, y), ha="center", va="center", fontsize=10, zorder=4,
                    bbox=dict(boxstyle="round,pad=0.16", fc="white", ec="none", alpha=0.85))
ax.set_xlim(-1.05, 1.05); ax.set_ylim(-1.05, 1.05)
ax.set_aspect("equal"); ax.axis("off")
ax.set_title("Prefix tree of GPT-2's 1,000 commonest word tokens")
plt.show()

## 11. Exercises

1. **Break the optimiser on purpose.** Delete the `riemannian_rescale` line from
   `train` and refit at 2-D. Then delete `project_to_ball` instead. Report what
   each failure looks like in the loss and in the largest radius.

2. **Find the crossover exactly.** The ladder brackets Euclidean between 20 and 50
   dimensions. Bisect it, with three seeds per point, and report the dimension at
   which Euclidean first matches 2-D hyperbolic on MAP rather than mean rank. Then
   repeat at `lr=2.0, epochs=1000` and see the answer move.

3. **Deepen the tree.** `balanced_tree(2, depth)` builds a perfect binary tree.
   Sweep depth from 4 to 10 with hyperbolic 2-D and Euclidean 10-D and plot mean
   rank against depth. Which curve bends?

4. **Give the flat baseline a fair fight.** The Euclidean runs use the same
   learning rate as the hyperbolic ones. Sweep the learning rate for Euclidean at
   10 dimensions and check whether the crossover moves.

5. **Embed something that is not a tree.** Build a random graph with the same node
   and edge count as the mammals closure and fit both geometries. The hyperbolic
   advantage should disappear, which is the point: the geometry encodes an
   assumption, and the assumption has to be true.

6. **Use the embedding.** Nearest neighbours under `poincare_distance` should
   return taxonomic siblings. Pick ten leaves, print their five nearest
   neighbours, and count how many share a parent.